### Loading the parquet already created

In [1]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
RAW_WEATHER = REPO_ROOT / "data" / "raw" / "weather" / "toronto_historical_weather_daily.parquet"
INTERIM_DIR = REPO_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

weather = pd.read_parquet(RAW_WEATHER)
print("weather shape:", weather.shape)
weather.head()

weather shape: (4383, 7)


,date,tavg,tmin,tmax,prcp,snow,wspd
0,2014-01-01,-11.1,-14.8,-8.8,0.0,0.21,18.0
1,2014-01-02,-17.3,-19.3,-14.9,2.5,2.24,23.1
2,2014-01-03,-17.8,-22.7,-10.9,0.0,0.00,20.3
3,2014-01-04,-4.6,-9.8,-1.2,0.2,0.35,28.2
4,2014-01-05,-0.6,-1.8,2.0,12.2,7.35,20.6


### Validate schema and types

In [2]:
required = ["date", "tavg", "tmin", "tmax", "prcp", "snow", "wspd"]
missing = [c for c in required if c not in weather.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Ensure date is date (not string)
weather["date"] = pd.to_datetime(weather["date"]).dt.date

# Ensure numeric
for c in ["tavg","tmin","tmax","prcp","snow","wspd"]:
    weather[c] = pd.to_numeric(weather[c], errors="coerce")

print(weather.isna().sum())
assert weather["date"].isna().sum() == 0

date    0
tavg    0
tmin    0
tmax    0
prcp    0
snow    0
wspd    0
dtype: int64


### Add simple derived flags

In [3]:
weather["freeze_day"] = (weather["tmin"] <= 0).astype(int)
weather["snow_day"]   = (weather["snow"] > 0).astype(int)
weather["rain_day"]   = (weather["prcp"] > 0).astype(int)

weather.head()

,date,tavg,tmin,tmax,prcp,snow,wspd,freeze_day,snow_day,rain_day
0,2014-01-01,-11.1,-14.8,-8.8,0.0,0.21,18.0,1,1,0
1,2014-01-02,-17.3,-19.3,-14.9,2.5,2.24,23.1,1,1,1
2,2014-01-03,-17.8,-22.7,-10.9,0.0,0.00,20.3,1,0,0
3,2014-01-04,-4.6,-9.8,-1.2,0.2,0.35,28.2,1,1,1
4,2014-01-05,-0.6,-1.8,2.0,12.2,7.35,20.6,1,1,1


### Save to interim

In [4]:
out_path = INTERIM_DIR / "weather_obs_daily.parquet"
weather.to_parquet(out_path, index=False)
print("Saved:", out_path, "rows:", len(weather))

Saved: C:\code\pyspark-playground\Covercheck-Toronto\data\interim\weather_obs_daily.parquet rows: 4383
